# Systemy uczące się - Zad. dom. 1: Minimalizacja ryzyka empirycznego
Celem zadania jest zaimplementowanie własnego drzewa decyzyjnego wykorzystującego idee minimalizacji ryzyka empirycznego.

### Autor rozwiązania
Uzupełnij poniższe informacje umieszczając swoje imię i nazwisko oraz numer indeksu:

In [ ]:
NAME = "Jakub Bilski"
ID = "155865"

## Twoja implementacja

Twoim celem jest uzupełnić poniższą klasę `TreeNode` tak by po wywołaniu `TreeNode.fit` tworzone było drzewo decyzyjne minimalizujące ryzyko empiryczne. Drzewo powinno wspierać problem klasyfikacji wieloklasowej (jak w przykładzie poniżej). Zaimplementowany algorytm nie musi (ale może) być analogiczny do zaprezentowanego na zajęciach algorytmu dla klasyfikacji. Wszelkie przejawy inwencji twórczej wskazane. **Pozostaw komenatrze w kodzie, które wyjaśniają Twoje rozwiązanie.**

Schemat oceniania:
- wynik na zbiorze Iris (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 10%: +40%,
- wynik na ukrytym zbiorze testowym 1 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 15%: +30%,
- wynik na ukrytym zbiorze testowym 2 (automatyczna ewaluacja) celność klasyfikacji >= prostego baseline'u + 5%: +30%.

Niedozwolone jest korzystanie z zewnętrznych bibliotek do tworzenia drzewa decyzyjnego (np. scikit-learn).
Możesz jedynie korzystać z biblioteki numpy.

#### Uwaga: Możesz dowolnie modyfikować elementy tego notebooka (wstawiać komórki i zmieniać kod), o ile będzie się w nim na koniec znajdowała kompletna implementacja klasy `TreeNode` w jednej komórce.

In [ ]:
import numpy as np

class TreeNode:
	def __init__(self):
		self.left = None # lewe dziecko (gdy warunek spełniony)
		self.right = None # prawe dziecko (gdy warunek niespełniony)
		self.feature_index = None # sprawdzana cecha
		self.threshold = None # porównywany próg
		self.leaf_value = None # zwracana klasa (w przypadku liścia)
		self.is_leaf = False

	def gini(self, y): # Gini impurity liczy prawdopodobieństwo, że pomylimy się przy losowym wyborze próbki z losowym przypisaniem jej etykiety
		if len(y) == 0:
			return 0.0
		_, counts = np.unique(y, return_counts=True) # unique zwraca dwie tablice: unikalne wartości i liczebności, interesują nas wyłącznie liczebności
		probabilities = counts / len(y)
		gini = 1.0 - np.sum(np.square(probabilities))
		return gini

	def find_best_split(self, data, target): # szukamy metodą brute-force splitu z max gain
		n_samples, n_features = data.shape
		parent_gini = self.gini(target) # punkt odniesienia - gini impurity węzła-rodzica
		best_gain = -np.inf
		best_feature = None
		best_threshold = None

		for feat_idx in range(n_features): # sprawdzamy każdą cechę osobno, bo nie wiemy która dzieli najlepiej
			col = data[:, feat_idx] # wybieramy kolumnę z daną cechą
			sorted_unique = np.unique(col)
			if len(sorted_unique) == 1:
				continue

			thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0 # obliczamy potencjalnie progi, czyli punkty w połowie między kolejnymi wartościami
			for thresh in thresholds:
				left_mask = col <= thresh # wyznaczamy tablice z wartościami boolowskimi - czy wartość poniżej lub powyżej progu?
				right_mask = col > thresh
				n_left = np.sum(left_mask) # sumowanie prawd
				n_right = np.sum(right_mask)
				if n_left == 0 or n_right == 0: # pomijamy podział, który wyszedł pusty
					continue

				gini_left = self.gini(target[left_mask]) # obliczamy gini impurity dla każdego dziecka osobno
				gini_right = self.gini(target[right_mask])
				weighted = (n_left * gini_left + n_right * gini_right) / n_samples # średnia ważona dzieci, bo liczebność dzieci może się drastycznie różnić
				gain = parent_gini - weighted # im większy, tym lepszy podział

				if gain > best_gain:
					best_gain = gain
					best_feature = feat_idx
					best_threshold = thresh

		return best_feature, best_threshold, best_gain

	def make_leaf(self, target): # tworzenie liścia - przypisanie klasy większościowej
		self.is_leaf = True
		classes, counts = np.unique(target, return_counts = True)
		self.leaf_value = classes[np.argmax(counts)] # argmax zwraca indeks klasy z największą liczebnością

	def fit(self, data: np.ndarray, target: np.ndarray, depth = 0) -> None: # rekurencyjna budowa drzewa,
		# sprawdzamy warunki stopu - jeśli tak, to budujemy liść
		# szukamy najlepszego podziału
		# dzielimy dane i budujemy dzieci

		if depth >= 10:
			self.make_leaf(target)
			return

		if len(np.unique(target)) == 1: # warunek stopu - wszystkie próbki jednej klasy i nie ma co dzielić
			self.make_leaf(target)
			return

		if len(target) < 2: # warunek stopu - za mało próbek do podziału
			self.make_leaf(target)
			return

		best_feat, best_thresh, best_gain = self.find_best_split(data, target)

		if best_feat is None or best_gain <= 0.000001: # warunek stopu - brak istotnej poprawy Gini impurity po podziale
			self.make_leaf(target)
			return

		self.feature_index = best_feat # zapisujemy decyzję w węźle
		self.threshold = best_thresh

		left_mask = data[:, best_feat] <= best_thresh
		self.left = TreeNode() # rekurencja - każde dziecko buduje swoje poddrzewo niezależnie
		self.right = TreeNode()

		self.left.fit(data[left_mask], target[left_mask], depth + 1)
		self.right.fit(data[~left_mask], target[~left_mask], depth + 1)

		"""
		Args:
			data (np.ndarray): macierz cech o wymiarach (n, m), gdzie n to liczba przykładów, a m to liczba cech
			target (np.ndarray): wektor klas o długości n, gdzie n to liczba przykładów
		"""

		# Poniej znajdziesz przykładowy "pseudo-kod" rozwiązania, nie musisz się go trzymać
		# (możesz zaimplementować to w inny sposób, jeżeli wolisz)
		#
		# Znajdź najlepszy podział x, y
		# if uzyskano poprawę funkcji celu (bądź inny, zaproponowany przez Ciebie warunek):
		# 	podziel dane na dwie części data_left i data_right, zgodnie z warunkiem
		# 	self.left = Node()
		# 	self.right = Node()
		# 	self.left.fit(data_left)
		# 	self.right.fit(data_right)
		# else:
		# 	obecny Node jest liściem, zapisz jego odpowiedź

	def predict_single(self, x): # predykcja na poziomie pojedynczej próbki - idziemy od korzenia do liści
		node = self
		while not node.is_leaf: # sprawdzamy warunek podziału, dopóki nie mamy liścia
			if x[node.feature_index] <= node.threshold: # wartość cechy <= próg - w którą stronę idziemy
				node = node.left
			else:
				node = node.right
		return node.leaf_value # liść zawiera klasę większościową

	def predict(self, data: np.ndarray) -> np.ndarray:
		"""
		Args:
			data (np.ndarray): macierz cech o wymiarach (n, m), gdzie n to liczba przykładów, a m to liczba cech

		Returns:
			np.ndarray: wektor przewidzoanych klas o długości n, gdzie n to liczba przykładów
		"""

		return np.array([self.predict_single(x) for x in data]) # dla każdego wiersza macierzy wywołuje się predict_single, wyniki zbierane są w tablicę


		# Poniżej znajdziesz przykładowy "pseudo-kod" rozwiązania, nie musisz się go trzymać
		# (możesz zaimplementować to w inny sposób, jeżeli wolisz),
		# ważne by metoda TreeNode.predict zwracała wektor przewidzianych klas
		#
		# Dla każdego przykładu w data:
		#   node = self
		#   if node nie jest liściem:
		#       if warunek podziału node jest spełniony:
		#           node = node.right
		#       else:
		#           node = node.left
		#   y_pred[i] = zwróć wartość node (liść)

## Przykład trenowanie i testowania drzewa

Później znajduje się przykład trenowania i testowania drzewa na zbiorze danych `iris`, który zawierający 150 próbek irysów, z czego każda próbka zawiera 4 atrybuty: długość i szerokość płatków oraz długość i szerokość działki kielicha. Każda próbka należy do jednej z trzech klas: `setosa`, `versicolor` lub `virginica`, które są zakodowane jak int.

Możesz go wykorzystać do testowania swojej implementacji. Możesz też zaimplementować własne testy lub użyć innych zbiorów danych, np. innych [zbiorów danych z scikit-learn](https://scikit-learn.org/stable/datasets/toy_dataset.html#toy-datasets).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.33, random_state=2024)

tree_model = TreeNode()
tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.88
